# Evaluation report EDA

This notebook is a read-only view over JSON reports generated by `eval run`. It uses pandas and matplotlib to inspect score trends, failures, citation quality, source coverage, weaknesses, and stage timings. It never writes reports, SQLite, indexes, Courses, or other application state.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

candidate = Path.cwd()
if (candidate / "pyproject.toml").is_file():
    ROOT = candidate
elif (candidate.parent / "pyproject.toml").is_file():
    ROOT = candidate.parent
else:
    raise RuntimeError(
        "Could not locate repository root: expected pyproject.toml in the current directory or its parent"
    )
REPORT_DIR = ROOT / "data" / "runs" / "eval"
REPORT_DIR  # read-only input location

In [ ]:
report_rows = []
for report_path in sorted(REPORT_DIR.glob("eval-*.json")):
    try:
        payload = json.loads(report_path.read_text(encoding="utf-8"))
    except (OSError, UnicodeDecodeError, json.JSONDecodeError):
        continue
    for item in payload.get("results", []):
        row = dict(item)
        row["report"] = report_path.name
        row["retrieval_passed"] = bool(
            (row.get("retrieval") or {}).get("passed", False)
        )
        row["citation_passed"] = bool((row.get("citations") or {}).get("passed", False))
        row["failure_count"] = len(row.get("failures") or [])
        row.update(
            {
                f"timing_{key}": value
                for key, value in (row.get("timings_ms") or {}).items()
            }
        )
        report_rows.append(row)
items = pd.DataFrame(report_rows)
items.head()

In [ ]:
if not items.empty:
    display(
        items.groupby("query_type")[
            ["passed", "retrieval_passed", "citation_passed"]
        ].mean()
    )
    display(
        items[["query_type", "status", "failure_count", "timing_total_ms"]].sort_values(
            "failure_count", ascending=False
        )
    )
else:
    print("No eval reports found. Run: uv run -m uni_rag_agent eval run")

In [ ]:
if not items.empty and {
    "timing_evidence_ms",
    "timing_answer_ms",
    "timing_total_ms",
}.issubset(items.columns):
    items[["timing_evidence_ms", "timing_answer_ms", "timing_total_ms"]].plot.box(
        title="Evaluation stage timings (ms)", rot=20
    )
    plt.ylabel("milliseconds")
    plt.show()
    items.groupby("query_type")["passed"].mean().sort_values().plot.barh(
        title="Pass rate by query type"
    )
    plt.xlabel("pass rate")
    plt.show()

## Safety boundary

Only report summaries and score fields are loaded. Raw evidence, raw model output, environment values, credentials, SQLite writes, index writes, and source-file traversal are intentionally absent from this notebook.